# Exercise 3 - Video Format Automation

This notebook automates the film festival workflow:
1. Examine each submitted film using **ffprobe**.
2. Detect which films do not satisfy the required specification.
3. Generate a **TXT report** with problematic fields.
4. Convert non-compliant films using **ffmpeg** and save copies with `_formatOK` suffix.

## FFmpeg/ffprobe setup (required)
This notebook requires **system executables** `ffmpeg` and `ffprobe` (from FFmpeg) to be installed and available on PATH.

### Windows
Install with one of the following commands in PowerShell:
- `winget install Gyan.FFmpeg`
- `choco install ffmpeg -y`

### macOS
- `brew install ffmpeg`

### Ubuntu/Debian
- `sudo apt update && sudo apt install -y ffmpeg`

After installation:
1. Restart VS Code/Jupyter kernel.
2. Re-run the setup cell below.
3. Confirm both tools are detected before running the pipeline.

## Target specification
- Container: `mp4`
- Video codec: `h264`
- Audio codec: `aac`
- Frame rate: `25 fps`
- Aspect ratio: `16:9`
- Resolution: `640 x 360`
- Video bitrate: `2-5 Mb/s`
- Audio bitrate: up to `256 kb/s`
- Audio channels: `stereo (2 channels)`

In [11]:
import shutil
import sys


def check_ffmpeg_tools():
    ffprobe_bin = shutil.which("ffprobe")
    ffmpeg_bin = shutil.which("ffmpeg")

    print("Python:", sys.version.split()[0])
    print("ffprobe:", ffprobe_bin if ffprobe_bin else "NOT FOUND")
    print("ffmpeg:", ffmpeg_bin if ffmpeg_bin else "NOT FOUND")

    if ffprobe_bin and ffmpeg_bin:
        print("\nFFmpeg tools are available. You can run the remaining cells.")
        return ffprobe_bin, ffmpeg_bin

    install_help = (
        "\nFFmpeg is missing. Install it, restart kernel, and rerun this cell.\n"
        "Windows (PowerShell): winget install Gyan.FFmpeg  OR  choco install ffmpeg -y\n"
        "macOS: brew install ffmpeg\n"
        "Ubuntu/Debian: sudo apt update && sudo apt install -y ffmpeg"
    )
    raise EnvironmentError(install_help)


check_ffmpeg_tools()

Python: 3.12.1
ffprobe: C:\ffmpeg\bin\ffprobe.EXE
ffmpeg: C:\ffmpeg\bin\ffmpeg.EXE

FFmpeg tools are available. You can run the remaining cells.


('C:\\ffmpeg\\bin\\ffprobe.EXE', 'C:\\ffmpeg\\bin\\ffmpeg.EXE')

In [12]:
from pathlib import Path
import json
import shutil
import subprocess
from fractions import Fraction

BASE_DIR = Path(".")
INPUT_DIR = BASE_DIR / "Input Videos"
OUTPUT_DIR = BASE_DIR / "Output Videos"
REPORT_PATH = BASE_DIR / "EX3_format_report.txt"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

video_exts = {".mp4", ".mov", ".avi", ".mkv", ".mpeg", ".mpg", ".wmv", ".webm", ".m4v"}
video_files = sorted([p for p in INPUT_DIR.iterdir() if p.suffix.lower() in video_exts])
assert video_files, f"No input video files found in: {INPUT_DIR.resolve()}"

ffprobe_bin = shutil.which("ffprobe")
ffmpeg_bin = shutil.which("ffmpeg")
assert ffprobe_bin, "ffprobe not found. Install FFmpeg and ensure ffprobe is on PATH."
assert ffmpeg_bin, "ffmpeg not found. Install FFmpeg and ensure ffmpeg is on PATH."

TARGET = {
    "container": "mp4",
    "video_codec": "h264",
    "audio_codec": "aac",
    "fps": 25.0,
    "width": 640,
    "height": 360,
    "aspect_ratio": Fraction(16, 9),
    "video_bitrate_min": 2_000_000,
    "video_bitrate_max": 5_000_000,
    "audio_bitrate_max": 256_000,
    "audio_channels": 2,
}

print("Found videos:")
for path in video_files:
    print("-", path.name)
print("\nffprobe:", ffprobe_bin)
print("ffmpeg:", ffmpeg_bin)

Found videos:
- Cosmos_War_of_the_Planets.mp4
- Last_man_on_earth_1964.mov
- The_Gun_and_the_Pulpit.avi
- The_Hill_Gang_Rides_Again.mp4
- Voyage_to_the_Planet_of_Prehistoric_Women.mp4

ffprobe: C:\ffmpeg\bin\ffprobe.EXE
ffmpeg: C:\ffmpeg\bin\ffmpeg.EXE


In [13]:
def run_ffprobe_json(video_path: Path):
    cmd = [
        ffprobe_bin,
        "-v",
        "error",
        "-print_format",
        "json",
        "-show_streams",
        "-show_format",
        str(video_path),
    ]
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    return json.loads(result.stdout)


def parse_fraction(text_value, default=0.0):
    if not text_value or text_value == "0/0":
        return float(default)
    try:
        return float(Fraction(text_value))
    except Exception:
        return float(default)


def safe_int(text_value, default=0):
    try:
        return int(text_value)
    except Exception:
        return default


def extract_metadata(video_path: Path):
    data = run_ffprobe_json(video_path)
    format_info = data.get("format", {})
    streams = data.get("streams", [])

    video_stream = next((s for s in streams if s.get("codec_type") == "video"), None)
    audio_stream = next((s for s in streams if s.get("codec_type") == "audio"), None)

    if video_stream is None:
        raise ValueError(f"No video stream found in {video_path.name}")

    width = safe_int(video_stream.get("width"), 0)
    height = safe_int(video_stream.get("height"), 0)

    fps = parse_fraction(video_stream.get("avg_frame_rate"), 0.0)
    if fps == 0.0:
        fps = parse_fraction(video_stream.get("r_frame_rate"), 0.0)

    v_codec = (video_stream.get("codec_name") or "").lower()
    a_codec = (audio_stream.get("codec_name") or "").lower() if audio_stream else ""

    v_bitrate = safe_int(video_stream.get("bit_rate"), 0)
    if v_bitrate == 0:
        v_bitrate = safe_int(format_info.get("bit_rate"), 0)

    a_bitrate = safe_int(audio_stream.get("bit_rate"), 0) if audio_stream else 0
    channels = safe_int(audio_stream.get("channels"), 0) if audio_stream else 0

    format_name = (format_info.get("format_name") or "").lower()
    container_ok = "mp4" in format_name

    meta = {
        "file": video_path.name,
        "path": video_path,
        "container": "mp4" if container_ok else format_name,
        "container_raw": format_name,
        "video_codec": v_codec,
        "audio_codec": a_codec,
        "fps": fps,
        "width": width,
        "height": height,
        "aspect_ratio": (width / height) if height else 0,
        "video_bitrate": v_bitrate,
        "audio_bitrate": a_bitrate,
        "audio_channels": channels,
    }
    return meta


def evaluate_compliance(meta):
    problems = []

    if "mp4" not in meta["container_raw"]:
        problems.append("container")

    if meta["video_codec"] != TARGET["video_codec"]:
        problems.append("video codec")

    if meta["audio_codec"] != TARGET["audio_codec"]:
        problems.append("audio codec")

    if abs(meta["fps"] - TARGET["fps"]) > 0.15:
        problems.append("frame rate")

    if meta["width"] != TARGET["width"] or meta["height"] != TARGET["height"]:
        problems.append("resolution")

    target_ar = float(TARGET["aspect_ratio"])
    if abs(meta["aspect_ratio"] - target_ar) > 0.02:
        problems.append("aspect ratio")

    if not (TARGET["video_bitrate_min"] <= meta["video_bitrate"] <= TARGET["video_bitrate_max"]):
        problems.append("video bit rate")

    if meta["audio_bitrate"] <= 0 or meta["audio_bitrate"] > TARGET["audio_bitrate_max"]:
        problems.append("audio bit rate")

    if meta["audio_channels"] != TARGET["audio_channels"]:
        problems.append("audio channels")

    return problems

In [14]:
def metadata_line(meta):
    return (
        f"container={meta['container_raw']} | vcodec={meta['video_codec']} | acodec={meta['audio_codec']} | "
        f"fps={meta['fps']:.3f} | res={meta['width']}x{meta['height']} | "
        f"vbitrate={meta['video_bitrate']} | abitrate={meta['audio_bitrate']} | channels={meta['audio_channels']}"
    )


audit_rows = []
non_compliant_rows = []

for video_path in video_files:
    meta = extract_metadata(video_path)
    problems = evaluate_compliance(meta)
    row = {"meta": meta, "problems": problems}
    audit_rows.append(row)
    if problems:
        non_compliant_rows.append(row)

print("Audit summary")
for row in audit_rows:
    tag = "OK" if not row["problems"] else "NOT OK"
    print(f"- {row['meta']['file']}: {tag}")

report_lines = []
report_lines.append("Exercise 3 - Video Format Compliance Report")
report_lines.append("=" * 46)
report_lines.append("")
report_lines.append(f"Total videos examined: {len(audit_rows)}")
report_lines.append(f"Videos with incorrect format: {len(non_compliant_rows)}")
report_lines.append("")

if not non_compliant_rows:
    report_lines.append("All videos satisfy the target specification.")
else:
    report_lines.append("Videos that do not meet the required format and problematic fields:")
    report_lines.append("")
    for row in non_compliant_rows:
        meta = row["meta"]
        fields = ", ".join(row["problems"])
        report_lines.append(f"- {meta['file']}")
        report_lines.append(f"  Problematic fields: {fields}")
        report_lines.append(f"  Metadata: {metadata_line(meta)}")
        report_lines.append("")

REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")
print(f"TXT report written to: {REPORT_PATH.resolve()}")

Audit summary
- Cosmos_War_of_the_Planets.mp4: NOT OK
- Last_man_on_earth_1964.mov: NOT OK
- The_Gun_and_the_Pulpit.avi: NOT OK
- The_Hill_Gang_Rides_Again.mp4: NOT OK
- Voyage_to_the_Planet_of_Prehistoric_Women.mp4: NOT OK
TXT report written to: C:\Users\Enzo\Desktop\SIM-UOL\Y3S1\CM3065 - Intelligent Signal Processing\EndTerm\Exercise3\EX3_format_report.txt


In [15]:
def converted_output_path(input_path: Path):
    return OUTPUT_DIR / f"{input_path.stem}_formatOK.mp4"


def build_ffmpeg_cmd(input_path: Path, output_path: Path):
    return [
        ffmpeg_bin,
        "-y",
        "-i",
        str(input_path),
        "-c:v",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        "-r",
        "25",
        "-s",
        "640x360",
        "-aspect",
        "16:9",
        "-b:v",
        "3500k",
        "-minrate",
        "2000k",
        "-maxrate",
        "5000k",
        "-bufsize",
        "5000k",
        "-c:a",
        "aac",
        "-b:a",
        "256k",
        "-ac",
        "2",
        str(output_path),
    ]


conversion_rows = []
for row in non_compliant_rows:
    input_path = row["meta"]["path"]
    output_path = converted_output_path(input_path)
    cmd = build_ffmpeg_cmd(input_path, output_path)

    print(f"Converting: {input_path.name} -> {output_path.name}")
    subprocess.run(cmd, check=True, capture_output=True, text=True)

    converted_meta = extract_metadata(output_path)
    converted_problems = evaluate_compliance(converted_meta)
    conversion_rows.append(
        {
            "input": input_path.name,
            "output": output_path.name,
            "problems_after": converted_problems,
            "meta_after": converted_meta,
        }
    )

print("\nConversion summary")
if not conversion_rows:
    print("No conversion required. All input videos were already compliant.")
else:
    for c in conversion_rows:
        ok = len(c["problems_after"]) == 0
        print(f"- {c['output']}: {'OK' if ok else 'NOT OK'}")
        if not ok:
            print("  Remaining issues:", ", ".join(c["problems_after"]))

all_converted_ok = all(len(c["problems_after"]) == 0 for c in conversion_rows)
if conversion_rows:
    assert all_converted_ok, "Some converted files still do not satisfy target format."

Converting: Cosmos_War_of_the_Planets.mp4 -> Cosmos_War_of_the_Planets_formatOK.mp4
Converting: Last_man_on_earth_1964.mov -> Last_man_on_earth_1964_formatOK.mp4
Converting: The_Gun_and_the_Pulpit.avi -> The_Gun_and_the_Pulpit_formatOK.mp4
Converting: The_Hill_Gang_Rides_Again.mp4 -> The_Hill_Gang_Rides_Again_formatOK.mp4
Converting: Voyage_to_the_Planet_of_Prehistoric_Women.mp4 -> Voyage_to_the_Planet_of_Prehistoric_Women_formatOK.mp4

Conversion summary
- Cosmos_War_of_the_Planets_formatOK.mp4: OK
- Last_man_on_earth_1964_formatOK.mp4: OK
- The_Gun_and_the_Pulpit_formatOK.mp4: OK
- The_Hill_Gang_Rides_Again_formatOK.mp4: OK
- Voyage_to_the_Planet_of_Prehistoric_Women_formatOK.mp4: OK
